In [2]:
"""
CNN baseline (ResNet50) for the PlantVillage dataset (abdallahalidev).
This dataset has NO train/valid split - images sit under color/<38 classes>/.
So we load the 'color' folder and make an 80/20 train/validation split in code.
Reports Accuracy, Precision, Recall, F1, and AUC.
"""

import os, json, random
import numpy as np
import torch
import torch.nn as nn
import timm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

# ---------------- config ----------------
DATA_ROOT  = "/kaggle/input"     # searched automatically for the variant folder
VARIANT    = "color"             # "color" | "grayscale" | "segmented"
VAL_SPLIT  = 0.2                 # 20% of images used for validation
EPOCHS     = 5
BATCH_SIZE = 32
LR         = 3e-4
IMG_SIZE   = 224
SEED       = 42
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# ------------- find the image folder (handles Kaggle nesting) -------------
def find_dir(root, name):
    for path, subdirs, _ in os.walk(root):
        for d in subdirs:
            if d.lower() == name:
                return os.path.join(path, d)
    raise FileNotFoundError(f"'{name}' folder not found under {root}")

data_dir = find_dir(DATA_ROOT, VARIANT)

# ---------------- data ----------------
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
valid_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# same folder loaded twice (train gets augmentation, valid does not),
# then split by shuffled indices so the split is reproducible
full_train = datasets.ImageFolder(data_dir, train_tf)
full_valid = datasets.ImageFolder(data_dir, valid_tf)
indices = list(range(len(full_train)))
random.Random(SEED).shuffle(indices)
cut = int((1 - VAL_SPLIT) * len(indices))
train_ds = Subset(full_train, indices[:cut])
valid_ds = Subset(full_valid, indices[cut:])

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
classes  = full_train.classes
print(f"Device: {DEVICE} | classes: {len(classes)} | "
      f"train: {len(train_ds)} | valid: {len(valid_ds)}")

# ---------------- model ----------------
model     = timm.create_model("resnet50", pretrained=True, num_classes=len(classes)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# ---------------- evaluation ----------------
def evaluate():
    model.eval()
    probs, trues = [], []
    with torch.no_grad():
        for x, y in valid_dl:
            out = model(x.to(DEVICE))
            probs.append(torch.softmax(out, 1).cpu().numpy())
            trues.append(y.numpy())
    probs = np.concatenate(probs)
    trues = np.concatenate(trues)
    preds = probs.argmax(1)
    return {
        "accuracy":  accuracy_score(trues, preds),
        "precision": precision_score(trues, preds, average="macro", zero_division=0),
        "recall":    recall_score(trues, preds, average="macro", zero_division=0),
        "f1":        f1_score(trues, preds, average="macro", zero_division=0),
        "auc":       roc_auc_score(trues, probs, average="macro", multi_class="ovr"),
    }

# ---------------- training ----------------
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    m = evaluate()
    print(f"epoch {epoch}/{EPOCHS}  loss {total/len(train_dl):.4f}  "
          f"acc {m['accuracy']:.4f}  f1 {m['f1']:.4f}")

# ---------------- final result ----------------
m = evaluate()
print("\n=== CNN (ResNet50) on PlantVillage - final results ===")
for k, v in m.items():
    print(f"  {k:10s}: {v:.4f}")

torch.save(model.state_dict(), "cnn_plantvillage.pt")
json.dump(classes, open("classes.json", "w"))
print("\nSaved model -> cnn_plantvillage.pt")

Device: cuda | classes: 38 | train: 43444 | valid: 10861


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

epoch 1/5  loss 0.6515  acc 0.9779  f1 0.9688
epoch 2/5  loss 0.1478  acc 0.9874  f1 0.9824
epoch 3/5  loss 0.1053  acc 0.9887  f1 0.9817
epoch 4/5  loss 0.0887  acc 0.9894  f1 0.9860
epoch 5/5  loss 0.0750  acc 0.9936  f1 0.9908

=== CNN (ResNet50) on PlantVillage - final results ===
  accuracy  : 0.9936
  precision : 0.9910
  recall    : 0.9910
  f1        : 0.9908
  auc       : 0.9999

Saved model -> cnn_plantvillage.pt


In [3]:
"""
ViT baseline (ViT-Base/16) for the PlantVillage dataset (abdallahalidev).
This dataset has NO train/valid split - images sit under color/<38 classes>/.
So we load the 'color' folder and make an 80/20 train/validation split in code.
Reports Accuracy, Precision, Recall, F1, and AUC.
"""

import os, json, random
import numpy as np
import torch
import torch.nn as nn
import timm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

# ---------------- config ----------------
DATA_ROOT  = "/kaggle/input"     # searched automatically for the variant folder
VARIANT    = "color"             # "color" | "grayscale" | "segmented"
VAL_SPLIT  = 0.2                 # 20% of images used for validation
EPOCHS     = 5
BATCH_SIZE = 32
LR         = 3e-4
IMG_SIZE   = 224
SEED       = 42
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# ------------- find the image folder (handles Kaggle nesting) -------------
def find_dir(root, name):
    for path, subdirs, _ in os.walk(root):
        for d in subdirs:
            if d.lower() == name:
                return os.path.join(path, d)
    raise FileNotFoundError(f"'{name}' folder not found under {root}")

data_dir = find_dir(DATA_ROOT, VARIANT)

# ---------------- data ----------------
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
valid_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# same folder loaded twice (train gets augmentation, valid does not),
# then split by shuffled indices so the split is reproducible
full_train = datasets.ImageFolder(data_dir, train_tf)
full_valid = datasets.ImageFolder(data_dir, valid_tf)
indices = list(range(len(full_train)))
random.Random(SEED).shuffle(indices)
cut = int((1 - VAL_SPLIT) * len(indices))
train_ds = Subset(full_train, indices[:cut])
valid_ds = Subset(full_valid, indices[cut:])

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
classes  = full_train.classes
print(f"Device: {DEVICE} | classes: {len(classes)} | "
      f"train: {len(train_ds)} | valid: {len(valid_ds)}")

# ---------------- model ----------------
model     = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=len(classes)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# ---------------- evaluation ----------------
def evaluate():
    model.eval()
    probs, trues = [], []
    with torch.no_grad():
        for x, y in valid_dl:
            out = model(x.to(DEVICE))
            probs.append(torch.softmax(out, 1).cpu().numpy())
            trues.append(y.numpy())
    probs = np.concatenate(probs)
    trues = np.concatenate(trues)
    preds = probs.argmax(1)
    return {
        "accuracy":  accuracy_score(trues, preds),
        "precision": precision_score(trues, preds, average="macro", zero_division=0),
        "recall":    recall_score(trues, preds, average="macro", zero_division=0),
        "f1":        f1_score(trues, preds, average="macro", zero_division=0),
        "auc":       roc_auc_score(trues, probs, average="macro", multi_class="ovr"),
    }

# ---------------- training ----------------
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    m = evaluate()
    print(f"epoch {epoch}/{EPOCHS}  loss {total/len(train_dl):.4f}  "
          f"acc {m['accuracy']:.4f}  f1 {m['f1']:.4f}")

# ---------------- final result ----------------
m = evaluate()
print("\n=== ViT (ViT-Base/16) on PlantVillage - final results ===")
for k, v in m.items():
    print(f"  {k:10s}: {v:.4f}")

torch.save(model.state_dict(), "vit_plantvillage.pt")
json.dump(classes, open("classes.json", "w"))
print("\nSaved model -> vit_plantvillage.pt")

Device: cuda | classes: 38 | train: 43444 | valid: 10861


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

epoch 1/5  loss 0.9498  acc 0.9268  f1 0.8958
epoch 2/5  loss 0.3739  acc 0.9036  f1 0.8770
epoch 3/5  loss 0.2763  acc 0.9405  f1 0.9124
epoch 4/5  loss 0.2396  acc 0.9525  f1 0.9376
epoch 5/5  loss 0.2138  acc 0.9525  f1 0.9361

=== ViT (ViT-Base/16) on PlantVillage - final results ===
  accuracy  : 0.9525
  precision : 0.9515
  recall    : 0.9295
  f1        : 0.9361
  auc       : 0.9997

Saved model -> vit_plantvillage.pt
